# Pratica — Lezione 7: completa la rotta POST /chatbot di Flask

Nel vero progetto guidato di IBM (`LLM_application_chatbot-main`), il backend Flask
(`app.py`) è già completo, ma il frontend (`static/script.js`) lascia l'endpoint del
chatbot non configurato (`const url = 'www.example.com';`) come esercizio per lo studente.
In questa pratica risolverai lo stesso tipo di "buco" ma lato server: completerai tu
stesso la funzione che gestisce `POST /chatbot`, e poi correggerai il
placeholder del frontend.

## Esercizio 1 — Completa `handle_prompt()`

Qui sotto trovi l'impalcatura di un server Flask con Blenderbot già caricato (come nel
notebook di lezione). Completa i `TODO` della funzione `handle_prompt()` in modo che la rotta
`POST /chatbot` funzioni dall'inizio alla fine: leggere il prompt dal JSON ricevuto, costruire lo
storico, tokenizzare, generare la risposta, aggiornare lo storico e restituire un JSON con la
risposta.

In [ ]:
!pip install -q flask flask-cors requests

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "facebook/blenderbot-400M-distill"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("Modello caricato.")

In [ ]:
import threading
import time
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

conversation_history = []

@app.route('/', methods=['GET'])
def home():
    return "Server Flask del chatbot attivo. Usa POST /chatbot con {'prompt': '...'}"

@app.route('/chatbot', methods=['POST'])
def handle_prompt():
    # TODO 1: leggi il JSON della richiesta con request.get_json() e salva
    #         il campo "prompt" in input_text
    input_text = None  # <-- sostituisci questo

    # TODO 2: costruisci la stringa dello storico unendo conversation_history con "\n"
    history = None  # <-- sostituisci questo

    # TODO 3: tokenizza history + input_text con tokenizer.encode_plus(...)
    #         (usa return_tensors="pt")
    inputs = None  # <-- sostituisci questo

    # TODO 4: genera la risposta con model.generate(**inputs, max_new_tokens=60)
    outputs = None  # <-- sostituisci questo

    # TODO 5: decodifica outputs[0] con tokenizer.decode(..., skip_special_tokens=True)
    #         e rimuovi gli spazi extra con .strip()
    response = None  # <-- sostituisci questo

    # TODO 6: aggiungi input_text e poi response a conversation_history

    return jsonify({"response": response})

def run_flask_app():
    app.run(port=5000, use_reloader=False)

server_thread = threading.Thread(target=run_flask_app, daemon=True)
server_thread.start()
time.sleep(3)
print("Server Flask in esecuzione su http://127.0.0.1:5000")

In [ ]:
import requests

# Questa cella funzionerà solo dopo aver completato i TODO della cella precedente
resp = requests.post(
    "http://127.0.0.1:5000/chatbot",
    json={"prompt": "Ciao, come stai?"}
)
print("Status:", resp.status_code)
print("Risposta:", resp.json())

<details>
<summary>💡 Vedi soluzione</summary>

```python
@app.route('/chatbot', methods=['POST'])
def handle_prompt():
    data = request.get_json()
    input_text = data['prompt']

    history = "\n".join(conversation_history)

    inputs = tokenizer.encode_plus(history, input_text, return_tensors="pt")

    outputs = model.generate(**inputs, max_new_tokens=60)

    response = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    conversation_history.append(input_text)
    conversation_history.append(response)

    return jsonify({"response": response})
```
</details>

## Esercizio 2 — Correggi l'endpoint del frontend

Questo è il bug/esercizio reale del progetto guidato di IBM: in `static/script.js`, la funzione
che invia il messaggio dell'utente al backend punta a un endpoint fittizio:

```javascript
async function makePostRequest(msg) {
  const url = 'www.example.com';  // <-- placeholder non risolto
  const requestBody = { prompt: msg };
  const response = await fetch(url, {
    method: 'POST',
    headers: { 'Content-Type': 'application/json' },
    body: JSON.stringify(requestBody)
  });
  // ...
}
```

**Il tuo compito**: sostituisci `'www.example.com'` con l'URL reale del tuo backend Flask. Nel
progetto guidato originale, il server veniva avviato su un Cloud IDE e il suo URL pubblico
terminava in `/chatbot`; girando sulla tua macchina (o nel notebook di lezione), sarebbe
`http://127.0.0.1:5000/chatbot`.

```javascript
async function makePostRequest(msg) {
  const url = // TODO: sostituisci questa riga con l'URL reale del tuo backend + "/chatbot"
  // ...
}
```

<details>
<summary>💡 Vedi soluzione</summary>

```javascript
async function makePostRequest(msg) {
  const url = 'http://127.0.0.1:5000/chatbot';
  const requestBody = { prompt: msg };
  const response = await fetch(url, {
    method: 'POST',
    headers: { 'Content-Type': 'application/json' },
    body: JSON.stringify(requestBody)
  });
  // ...
}
```

Se il tuo server Flask girasse su un Cloud IDE o su un servizio come IBM Cloud Code Engine, la
URL sarebbe l'indirizzo pubblico di quel servizio terminato in `/chatbot`, non `127.0.0.1`.
</details>

Torna al [notebook di lezione](../notebooks_clase/clase07_chatbot_web_flask_IT.ipynb) o ripassa il
[Capitolo 11 del manuale](../manual/11_chatbot_web_flask_IT.md).